In [0]:
#Define the dataset location in Unity Catalog
file_path = "/Volumes/workspace/sudoku_bronze/raw_files/sudoku.csv"

# Read the CSV while preserving Sudoku boards as strings
sudoku_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(file_path)
)

#Inspect the DataFrame schema
sudoku_df.printSchema()

In [0]:
#Count the total number of records
total_records = sudoku_df.count()

print(f"Total records: {total_records:,}")

#Preview the first five records
display(sudoku_df.limit(5))

In [0]:
#Call libraries
from pyspark.sql import functions as F

#Add ingestion metadata while preserving source columns
bronze_df = (
    sudoku_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit(file_path))
)

#Preview the Bronze DataFrame
display(bronze_df.limit(5))

In [0]:
#Define the Bronze table
bronze_table = "workspace.sudoku_bronze.raw_games"

#Save the DataFrame as a managed Delta table
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Bronze table created: {bronze_table}")

In [0]:
#Read the persisted Bronze table
bronze_table_df = spark.table(bronze_table)

#Validate the number of records
print(f"Bronze records: {bronze_table_df.count():,}")

#Inspect the schema
bronze_table_df.printSchema()